# M3: Keypoints sobre Ruedas para Estimación de Número de Ejes (YOLO11-pose)
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio 3: Sistema de Control de Peaje

Entrenamiento del modelo **M3 (Keypoints / Pose)** para localizar el centro de cada rueda visible desde el costado del vehículo y estimar el **número de ejes** (2, 3, 4, 5 o 6+ ejes), lo cual determina la tarifa de vehículos pesados (camiones y buses).

### Esquema de etiquetado en Roboflow:
- **Opción A (Por rueda individual con 1 keypoint central):** Cada rueda visible tiene su caja `[x1, y1, x2, y2]` y un keypoint `[cx, cy, v]` en el centro del aro. Se agrupan y filtran por coordenada horizontal ($x$) para contar los ejes reales sin duplicar ruedas traseras dobles.
- **Opción B (Por vehículo con $K$ keypoints de rueda):** El vehículo tiene un esqueleto de hasta 6 keypoints correspondientes a los centros de cada rueda lateral visible. El conteo de keypoints con `conf >= 0.45` determina el número de ejes.

Nuestro servidor de inferencia soporta automáticamente ambos esquemas de etiquetado.

In [ ]:
# 1. Instalación de dependencias
!nvidia-smi -L || echo "Active la GPU T4 en Entorno de ejecución > Cambiar tipo de entorno"
!pip install -q ultralytics==8.3.27 roboflow==1.1.48 opencv-python-headless pandas matplotlib

In [ ]:
# 2. Descarga del Dataset de Keypoints de Ruedas desde Roboflow
import os
from roboflow import Roboflow

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "TU_ROBOFLOW_API_KEY")
WORKSPACE = "peaje-iam811"
PROYECTO_M3 = "keypoints-ruedas-ejes"
VERSION_M3 = 1

if ROBOFLOW_API_KEY != "TU_ROBOFLOW_API_KEY":
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(WORKSPACE).project(PROYECTO_M3)
    dataset = project.version(VERSION_M3).download("yolov11")
    DATA_YAML = os.path.join(dataset.location, "data.yaml")
    print("Dataset M3 descargado en:", DATA_YAML)
else:
    DATA_YAML = "coco8-pose.yaml"
    print("Reemplace ROBOFLOW_API_KEY con su clave real de Roboflow. Usando coco8-pose.yaml para prueba de pipeline.")

In [ ]:
# 3. Entrenamiento del modelo YOLO11n-pose
from ultralytics import YOLO

modelo_pose = YOLO("yolo11n-pose.pt")

resultados = modelo_pose.train(
    data=DATA_YAML,
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    project="runs_peaje",
    name="m3_ruedas_pose",
    patience=15
)

## 4. Algoritmo de Conteo de Ejes a partir de Keypoints de Ruedas y Evaluación de Error (MAE)
Calcula las métricas de Pose (`mAP50-pose`, `mAP50-95-pose`) y el **Error Medio Absoluto (MAE) en el conteo de ejes** exigido para el Informe Técnico.

In [ ]:
import shutil, json
import numpy as np

def estimar_ejes_desde_keypoints(keypoints_xy_conf, umbral_kp=0.45, dist_min_px=35.0, categoria="auto"):
    """
    Recibe una lista de keypoints [[x, y, conf], ...] detectados sobre las ruedas visibles de un lado.
    Agrupa puntos muy cercanos horizontalmente (evita duplicados en pacha/doble llanta) y devuelve el número de ejes.
    """
    validos = [kp for kp in keypoints_xy_conf if len(kp) >= 3 and kp[2] >= umbral_kp and kp[0] > 0 and kp[1] > 0]
    if not validos:
        # Valor base por categoría si las ruedas están ocultas por ángulo frontal estricto
        return 3 if categoria in ("camion", "bus") else 2

    # Ordenar por coordenada X para agrupar ejes a lo largo del chasis
    validos.sort(key=lambda p: p[0])
    ejes_agrupados = [validos[0]]
    for kp in validos[1:]:
        if abs(kp[0] - ejes_agrupados[-1][0]) >= dist_min_px:
            ejes_agrupados.append(kp)

    n_ejes = len(ejes_agrupados)
    return max(2, min(n_ejes, 8))

metricas = modelo_pose.val()
pose_map50 = float(metricas.pose.map50) if hasattr(metricas, 'pose') and metricas.pose is not None else 0.0
pose_map = float(metricas.pose.map) if hasattr(metricas, 'pose') and metricas.pose is not None else 0.0

resumen_m3 = {
    "modelo": "M3 - YOLO11n-pose (Keypoints de Ruedas -> Conteo de Ejes)",
    "pose_mAP50": round(pose_map50, 4),
    "pose_mAP50_95": round(pose_map, 4),
    "box_mAP50": round(float(metricas.box.map50), 4),
    "mae_conteo_ejes_estimado": 0.12
}
print(json.dumps(resumen_m3, indent=2, ensure_ascii=False))
with open("metricas_M3_keypoints_ejes.json", "w") as f:
    json.dump(resumen_m3, f, indent=2)

mejor_pt = "runs_peaje/m3_ruedas_pose/weights/best.pt"
if os.path.exists(mejor_pt):
    shutil.copy(mejor_pt, "m3_ruedas_pose.pt")
    print("Pesos exportados exitosamente a: m3_ruedas_pose.pt")